# Create round_info.csv

Builds `round_info.csv`, the per-round series / HAL config / data folder table
that notebook 04 turns into the Dave recipe, from the HAL configs (01) and the
positions (02b).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave      import create_round_info, add_frame_table_column, match_bits_hal_configs
from MERci.acquisition.pipeline_config import load_pipeline_config

PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Positions files are named with POSITIONS_TAG.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────
MICROSCOPE  = PIPELINE_CONFIG.microscope   # microscope identifier


# Optional: spread the raw data over several drives. HYB_DRIVE_GROUPS lists
# consecutive (count, drive) blocks over the hyb rounds; counts sum to N_HYBS.
# CELLS_DRIVE moves the cells round. [] / None = everything on SAMPLE_DIR's drive.
#   HYB_DRIVE_GROUPS = [(6, "Y:"), (6, "Z:"), (6, "Y:"), (7, "Z:")]
#   CELLS_DRIVE      = "Z:"
HYB_DRIVE_GROUPS = []
CELLS_DRIVE      = None

# HAL configs written by notebook 01
bits_hal_configs    = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
cells_hal_configs   = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))

print("Available HAL configs in settings/:")
for p in sorted(SETTINGS_DIR.glob("hal-config-*.xml")):
    print(f"  {p.name}")

# None = one bits config per hyb round, matched to its colours in
# round_bit_color. A file name forces one config for every round.
BITS_HAL_CONFIG    = None
CELLS_HAL_CONFIG   = cells_hal_configs[0].name   if cells_hal_configs   else "hal-config-mf3-cells.xml"

print(f"Cells   HAL config : {CELLS_HAL_CONFIG}")

## Round – bit – color mapping

Read from this pipeline's round_bit_color CSV (pipeline.yaml's
`dataorganization.round_bit_color_csv`). Sets `N_HYBS` (the highest round) and
is saved to `metadata/round_bit_color_map.csv` for the later notebooks.

In [ ]:
# round : hyb/bit index (1-based) = bits movie series number (hal-{mic}_01, ...),
#         not the Dave imaging round
# bit   : bit number     |     color : excitation wavelength (nm)
round_bit_color = PIPELINE_CONFIG.round_bit_color

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)

N_HYBS = int(rbc_df["round"].max())   # number of bits rounds, derived from the mapping
print(f"Saved: {rbc_path}")
print(f"N_HYBS (from mapping): {N_HYBS}")
print(rbc_df.to_string(index=False))

In [ ]:
bits_hal_config = BITS_HAL_CONFIG or match_bits_hal_configs(rbc_df, bits_hal_configs, METADATA_DIR)
print(f"Bits HAL config per hyb round: {bits_hal_config}\n")

round_info = create_round_info(
    microscope       = MICROSCOPE,
    n_bits           = N_HYBS,
    bits_hal_config  = bits_hal_config,
    cells_hal_config = CELLS_HAL_CONFIG,
    sample_dir       = SAMPLE_DIR,
    positions_txt    = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt",
    hyb_drive_groups = HYB_DRIVE_GROUPS or None,
    cells_drive      = CELLS_DRIVE,
)

# Each round's frame table, for create_merlin_files. Edit a row here or in the
# CSV to override.
round_info = add_frame_table_column(round_info, SETTINGS_DIR, METADATA_DIR)

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")